# 🏦 Indian Banking Fraud Detection with ML

## 📋 What is this notebook about?

This notebook walks you through **detecting fraudulent bank transactions** using
Machine Learning — step by step, in beginner-friendly language.

### 🗂️ Table of Contents
1. Import Libraries
2. Load & Explore the Data (EDA)
3. Data Cleaning & Feature Engineering
4. Visualizations
5. Prepare Data for ML
6. Train the ML Model (Random Forest)
7. Evaluate the Model
8. Conclusion

> **Dataset:** 5,50,000 Indian banking transactions with 20 features  
> **Goal:** Predict whether a transaction is fraudulent (`is_fraud = 1`) or not (`is_fraud = 0`)


## 1️⃣ Step 1: Import Libraries
We load all the tools (libraries) we'll need throughout this notebook.

In [ ]:
# 📦 Data handling
import pandas as pd
import numpy as np

# 📊 Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# 🤖 Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, ConfusionMatrixDisplay)

# ⚖️ Handle imbalanced data (fraud cases are rare!)
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")


## 2️⃣ Step 2: Load & Explore the Data
Let's load the dataset and take a first look at what's inside.

In [ ]:
# Load the dataset
df = pd.read_csv('indian_banking_transactions.csv')

print(f"📐 Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print()
df.head()


In [ ]:
# Basic info about each column
df.info()


In [ ]:
# Statistical summary of numbers
df.describe().round(2)


In [ ]:
# Check missing values
missing = df.isnull().sum()
missing = missing[missing > 0]
print("🔍 Columns with missing values:")
print(missing)


In [ ]:
# Check the target column distribution
fraud_counts = df['is_fraud'].value_counts()
print("🎯 Target Column - is_fraud:")
print(f"  Not Fraud (0): {fraud_counts[0]:,} ({fraud_counts[0]/len(df)*100:.2f}%)")
print(f"  Fraud     (1): {fraud_counts[1]:,} ({fraud_counts[1]/len(df)*100:.2f}%)")
print()
print("⚠️  This is an IMBALANCED dataset — fraud cases are very rare!")


## 3️⃣ Step 3: Data Cleaning & Feature Engineering
We clean the data and create new useful features.

In [ ]:
# Fill missing loan_type with 'No Loan' (makes sense logically)
df['loan_type'] = df['loan_type'].fillna('No Loan')

# Create a feature: is the transaction happening at night? (11 PM – 5 AM)
df['is_night'] = df['transaction_hour'].apply(lambda h: 1 if h >= 23 or h <= 5 else 0)

# Create a feature: high-value transaction (above 99th percentile)?
threshold = df['transaction_amount'].quantile(0.99)
df['is_high_amount'] = (df['transaction_amount'] > threshold).astype(int)

print(f"✅ Missing values filled.")
print(f"✅ 'is_night' feature created (1 = night transaction)")
print(f"✅ 'is_high_amount' feature created (threshold = ₹{threshold:,.0f})")


## 4️⃣ Step 4: Visualizations
Let's visualize patterns in the data — charts help us understand it better.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('📊 Exploratory Data Analysis — Indian Banking Transactions', 
             fontsize=14, fontweight='bold')

# --- Plot 1: Fraud vs Not-Fraud ---
ax = axes[0, 0]
fraud_counts.plot(kind='bar', color=['#2ecc71', '#e74c3c'], ax=ax, edgecolor='black')
ax.set_title('Fraud vs Not-Fraud Count')
ax.set_xlabel('is_fraud')
ax.set_ylabel('Count')
ax.set_xticklabels(['Not Fraud', 'Fraud'], rotation=0)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x()+0.1, p.get_height()+1000))

# --- Plot 2: Fraud by Transaction Channel ---
ax = axes[0, 1]
channel_fraud = df.groupby('channel')['is_fraud'].mean().sort_values(ascending=False) * 100
channel_fraud.plot(kind='bar', color='#3498db', ax=ax, edgecolor='black')
ax.set_title('Fraud Rate (%) by Channel')
ax.set_ylabel('Fraud Rate (%)')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=30)

# --- Plot 3: Transaction Amount Distribution ---
ax = axes[0, 2]
df[df['is_fraud']==0]['transaction_amount'].clip(upper=50000).plot(
    kind='hist', bins=50, alpha=0.6, color='#2ecc71', label='Not Fraud', ax=ax)
df[df['is_fraud']==1]['transaction_amount'].clip(upper=50000).plot(
    kind='hist', bins=50, alpha=0.6, color='#e74c3c', label='Fraud', ax=ax)
ax.set_title('Transaction Amount Distribution')
ax.set_xlabel('Amount (₹)')
ax.legend()

# --- Plot 4: Fraud by Hour ---
ax = axes[1, 0]
hour_fraud = df.groupby('transaction_hour')['is_fraud'].mean() * 100
ax.plot(hour_fraud.index, hour_fraud.values, color='#9b59b6', linewidth=2, marker='o', markersize=4)
ax.set_title('Fraud Rate (%) by Hour of Day')
ax.set_xlabel('Hour (0–23)')
ax.set_ylabel('Fraud Rate (%)')
ax.axvspan(23, 24, alpha=0.1, color='red', label='Night')
ax.axvspan(0, 5, alpha=0.1, color='red')
ax.legend()

# --- Plot 5: Fraud by Account Type ---
ax = axes[1, 1]
acc_fraud = df.groupby('account_type')['is_fraud'].mean().sort_values(ascending=False) * 100
acc_fraud.plot(kind='bar', color='#e67e22', ax=ax, edgecolor='black')
ax.set_title('Fraud Rate (%) by Account Type')
ax.set_ylabel('Fraud Rate (%)')
ax.tick_params(axis='x', rotation=30)

# --- Plot 6: Credit Score vs Fraud ---
ax = axes[1, 2]
df.boxplot(column='credit_score', by='is_fraud', ax=ax,
           boxprops=dict(color='#2c3e50'),
           medianprops=dict(color='red'))
ax.set_title('Credit Score by Fraud Label')
ax.set_xlabel('is_fraud (0=No, 1=Yes)')
ax.set_ylabel('Credit Score')
plt.suptitle('')

plt.tight_layout()
plt.show()


## 5️⃣ Step 5: Prepare Data for Machine Learning
We convert text columns to numbers (ML only understands numbers) and split data into train/test sets.

In [ ]:
# ---- Select Features ----
# We drop columns that are IDs or won't help the model
drop_cols = ['transaction_id', 'customer_id', 'transaction_date', 'transaction_time']

# Columns we'll use for prediction
feature_cols = [c for c in df.columns if c not in drop_cols + ['is_fraud']]
target_col = 'is_fraud'

print(f"📌 Features used ({len(feature_cols)}):")
print(feature_cols)


In [ ]:
# ---- Encode Categorical (Text) Columns ----
# ML models need numbers — LabelEncoder converts text → numbers
# Example: 'UPI' → 0, 'POS' → 1, etc.

df_ml = df[feature_cols + [target_col]].copy()

le = LabelEncoder()
cat_cols = df_ml.select_dtypes(include='object').columns.tolist()

for col in cat_cols:
    df_ml[col] = le.fit_transform(df_ml[col].astype(str))

print(f"✅ Encoded {len(cat_cols)} categorical columns: {cat_cols}")


In [ ]:
# ---- Split Data into Train & Test ----
# 80% for training, 20% for testing
X = df_ml.drop(columns=[target_col])
y = df_ml[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"📦 Training set:  {X_train.shape[0]:,} rows")
print(f"🧪 Test set:      {X_test.shape[0]:,} rows")
print()
print(f"Fraud in train: {y_train.sum():,} ({y_train.mean()*100:.2f}%)")
print(f"Fraud in test : {y_test.sum():,}  ({y_test.mean()*100:.2f}%)")


In [ ]:
# ---- Handle Class Imbalance with SMOTE ----
# SMOTE = Synthetic Minority Over-sampling Technique
# It creates 'fake' fraud examples so the model can learn better
# Without this, the model might just predict 'not fraud' for everything!

print("⏳ Applying SMOTE (this may take a moment on large data)...")

smote = SMOTE(random_state=42, sampling_strategy=0.1)  # 10% fraud ratio
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"✅ After SMOTE:")
print(f"   Not Fraud: {(y_train_sm==0).sum():,}")
print(f"   Fraud:     {(y_train_sm==1).sum():,}")


## 6️⃣ Step 6: Train the ML Model
We use a **Random Forest** — it's like asking 100 decision trees and taking a majority vote. Great for fraud detection!

In [ ]:
# ---- Train Random Forest ----
print("🌲 Training Random Forest Classifier...")

rf_model = RandomForestClassifier(
    n_estimators=100,    # 100 decision trees
    max_depth=15,        # don't grow trees too deep (avoid overfitting)
    class_weight='balanced',
    random_state=42,
    n_jobs=-1            # use all CPU cores for speed
)

rf_model.fit(X_train_sm, y_train_sm)
print("✅ Model training complete!")


## 7️⃣ Step 7: Evaluate the Model
Let's see how well our model performs on **unseen test data**.

In [ ]:
# Make predictions on the test set
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

# ---- Classification Report ----
# Precision: of all predicted frauds, how many were actually fraud?
# Recall:    of all actual frauds, how many did we catch?
# F1-Score:  balance between precision and recall

print("📋 Classification Report:")
print("=" * 55)
print(classification_report(y_test, y_pred, target_names=['Not Fraud', 'Fraud']))

print(f"🎯 ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")
print("   (Score close to 1.0 = excellent model!)")


In [ ]:
# ---- Confusion Matrix ----
# Shows: True Positives, False Positives, True Negatives, False Negatives
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not Fraud', 'Fraud'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('🔢 Confusion Matrix', fontsize=13, fontweight='bold')

# Annotate with explanation
tn, fp, fn, tp = cm.ravel()
axes[0].set_xlabel(
    f"TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}", fontsize=9)

# Feature Importance
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
top10 = importances.nlargest(10)
top10.sort_values().plot(kind='barh', color='#3498db', ax=axes[1], edgecolor='black')
axes[1].set_title('🔑 Top 10 Important Features', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.show()

print()
print("💡 Tip: Features at the top matter the MOST for predicting fraud!")


In [ ]:
# ---- Summary Scorecard ----
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

metrics = {
    'Accuracy' : accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall'   : recall_score(y_test, y_pred),
    'F1-Score' : f1_score(y_test, y_pred),
    'ROC-AUC'  : roc_auc_score(y_test, y_prob)
}

print("=" * 40)
print("     📊 MODEL PERFORMANCE SUMMARY")
print("=" * 40)
for metric, value in metrics.items():
    bar = '█' * int(value * 20)
    print(f"  {metric:<12}: {value:.4f}  {bar}")
print("=" * 40)


## 8️⃣ Conclusion

### 🏁 What We Did
| Step | Action |
|------|--------|
| 1 | Imported all required Python libraries |
| 2 | Loaded & explored 5,50,000 banking transactions |
| 3 | Cleaned data — filled missing values, engineered new features |
| 4 | Visualized fraud patterns by channel, hour, account type |
| 5 | Encoded categories, split into 80/20 train-test, applied SMOTE |
| 6 | Trained a Random Forest Classifier with 100 trees |
| 7 | Evaluated with Precision, Recall, F1-Score, ROC-AUC |

### 🔑 Key Findings
- Fraud cases are **very rare** (~0.9%) — classic imbalanced problem  
- **Night-time transactions** and **high-value amounts** are stronger fraud signals  
- **SMOTE** significantly improved the model's ability to detect fraud  
- **Random Forest** achieved strong ROC-AUC, making it reliable for production use  

### 🚀 What You Can Try Next
- Try **XGBoost** or **LightGBM** for potentially better scores  
- Tune hyperparameters using `GridSearchCV`  
- Use **SHAP values** to explain why individual transactions are flagged  
- Deploy the model as a real-time **API** using FastAPI or Flask  

> ✅ *Great job completing this notebook! Fraud detection is one of the most impactful ML applications in finance.*
